# SocForge — Threat Hunting Notebook

**Lab**: SocForge SOC Lab  
**Date**: 2026-08-15  
**Analyste**: SOC SocForge  
**Source de données**: Wazuh OpenSearch (localhost:9200), export 2026-08-07

---

## Objectif

Recherche proactive de comportements suspects non détectés par les règles automatiques.  
Méthodologie : hypothèse → requête → analyse → règle ou conclusion.

---

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import requests
import urllib3
urllib3.disable_warnings()

# Configuration OpenSearch
OS_URL = 'https://127.0.0.1:9200'
OS_USER = 'admin'
OS_PASS = '<mot de passe du lab>'
INDEX = 'wazuh-alerts-*'

def query_os(body, size=500):
    """Requête OpenSearch, retourne DataFrame."""
    body['size'] = size
    r = requests.post(
        f'{OS_URL}/{INDEX}/_search',
        auth=(OS_USER, OS_PASS),
        json=body,
        verify=False,
        timeout=30
    )
    hits = r.json().get('hits', {}).get('hits', [])
    return pd.json_normalize([h['_source'] for h in hits])

print('OpenSearch client prêt.')

## Hunt 1 — PowerShell rare ou inhabituel

**Hypothèse** : Un attaquant utilisant PowerShell avec des commandes encodées ou inhabituelles laissera des traces dans les événements Sysmon (EventID 4104 / Script Block Logging).

**Source** : WIN01 Sysmon + PowerShell Script Block Logging  
**Règles attendues** : 100120, 100121, 100131

In [ ]:
# Hunt 1 : PowerShell suspect
query = {
    'query': {
        'bool': {
            'must': [
                {'term': {'agent.name': 'win01'}},
                {'terms': {'rule.id': ['100120', '100121', '100131']}}
            ]
        }
    },
    'sort': [{'@timestamp': {'order': 'desc'}}]
}

df_ps = query_os(query)
print(f'Alertes PowerShell trouvées : {len(df_ps)}')

if not df_ps.empty and 'rule.description' in df_ps.columns:
    print('\nDistribution par règle :')
    print(df_ps.groupby('rule.id')['rule.description'].first())
else:
    print('Colonnes disponibles :', list(df_ps.columns)[:10])

## Hunt 2 — Brute Force et authentifications anormales

**Hypothèse** : Une activité de brute force laisse une séquence d'échecs EventID 4625 suivie d'un succès 4624 (credential stuffing).

**Source** : DC01 Security Event Log  
**Règles attendues** : 100110, 100111, 60122

In [ ]:
# Hunt 2 : Brute Force
query_bf = {
    'query': {
        'bool': {
            'must': [
                {'term': {'agent.name': 'dc01'}},
                {'terms': {'rule.id': ['100110', '100111', '60122']}}
            ]
        }
    },
    'sort': [{'@timestamp': {'order': 'asc'}}]
}

df_bf = query_os(query_bf)
print(f'Alertes brute force trouvées : {len(df_bf)}')

if not df_bf.empty:
    print('\nPremières alertes :')
    cols = [c for c in ['timestamp', 'rule.id', 'rule.description', 'rule.level'] if c in df_bf.columns]
    print(df_bf[cols].head(10).to_string())

## Hunt 3 — Mouvements latéraux (Admin Shares)

**Hypothèse** : Accès aux partages administratifs (ADMIN$, C$) indique un mouvement latéral possible.

**Source** : WIN01 / DC01 EventID 5140  
**Règle** : 100140

In [ ]:
# Hunt 3 : Admin Shares
query_shares = {
    'query': {
        'bool': {
            'must': [{'term': {'rule.id': '100140'}}]
        }
    },
    'aggs': {
        'par_agent': {
            'terms': {'field': 'agent.name', 'size': 10}
        }
    }
}

df_shares = query_os(query_shares)
print(f'Alertes admin shares : {len(df_shares)}')

# Volume par agent
if not df_shares.empty and 'agent.name' in df_shares.columns:
    print('\nVolume par agent :')
    print(df_shares['agent.name'].value_counts())

## Hunt 4 — Persistence : Tâches planifiées et modifications de registre

**Hypothèse** : La création de tâches planifiées ou la modification de clés Run/RunOnce est un indicateur de persistence post-exploitation.

**Source** : WIN01 EventID 4698, Sysmon EventID 13  
**Règles** : 100153, 92302, 60642

In [ ]:
# Hunt 4 : Persistence
query_persist = {
    'query': {
        'bool': {
            'must': [
                {'term': {'agent.name': 'win01'}},
                {'terms': {'rule.id': ['100153', '92302', '60642', '92004']}}
            ]
        }
    },
    'sort': [{'@timestamp': {'order': 'asc'}}]
}

df_persist = query_os(query_persist)
print(f'Alertes persistence trouvées : {len(df_persist)}')

if not df_persist.empty:
    cols = [c for c in ['timestamp', 'rule.id', 'rule.description'] if c in df_persist.columns]
    print(df_persist[cols].to_string())

## Hunt 5 — Dump de credentials LSASS

**Hypothèse** : Un accès à LSASS (Local Security Authority) via Sysmon EventID 10 indique une tentative de vol de credentials.

**Source** : WIN01 Sysmon-10  
**Règle** : 100121 (niveau 14)

In [ ]:
# Hunt 5 : LSASS Access
query_lsass = {
    'query': {
        'bool': {
            'must': [
                {'term': {'agent.name': 'win01'}},
                {'term': {'rule.id': '100121'}},
                {'range': {'rule.level': {'gte': 12}}}
            ]
        }
    },
    'sort': [{'@timestamp': {'order': 'desc'}}]
}

df_lsass = query_os(query_lsass)
print(f'Alertes LSASS (niveau ≥ 12) : {len(df_lsass)}')

if not df_lsass.empty:
    print('\nDernières alertes critiques :')
    cols = [c for c in ['timestamp', 'rule.level', 'rule.description'] if c in df_lsass.columns]
    print(df_lsass[cols].head(5).to_string())

## Analyse des KPI — Timeline des alertes

In [ ]:
# Données MTTD réelles issues des 13 scénarios Purple Team
scenarios = [
    {'sc': 'SC-01', 'technique': 'T1059.001', 'mttd': 12, 'session': 1},
    {'sc': 'SC-02', 'technique': 'T1110',     'mttd': 8,  'session': 1},
    {'sc': 'SC-03', 'technique': 'T1046',     'mttd': 15, 'session': 1},
    {'sc': 'SC-04', 'technique': 'T1021.002', 'mttd': 23, 'session': 1},
    {'sc': 'SC-05', 'technique': 'T1003.001', 'mttd': 6,  'session': 1},
    {'sc': 'SC-06', 'technique': 'T1055',     'mttd': 9,  'session': 1},
    {'sc': 'SC-07', 'technique': 'T1547.001', 'mttd': 18, 'session': 1},
    {'sc': 'SC-08', 'technique': 'T1548.003', 'mttd': 11, 'session': 2},
    {'sc': 'SC-09', 'technique': 'T1053.003', 'mttd': 14, 'session': 2},
    {'sc': 'SC-10', 'technique': 'T1053.005', 'mttd': 84, 'session': 3},
    {'sc': 'SC-11', 'technique': 'T1078',     'mttd': 42, 'session': 3},
    {'sc': 'SC-12', 'technique': 'T1546.013', 'mttd': 49, 'session': 3},
]

df_kpi = pd.DataFrame(scenarios)
print('=== MTTD par scénario ===')
print(df_kpi[['sc', 'technique', 'mttd', 'session']].to_string())
print(f'\nMTTD global moyen : {df_kpi["mttd"].mean():.1f}s')
print(f'MTTD max : {df_kpi["mttd"].max()}s ({df_kpi.loc[df_kpi["mttd"].idxmax(), "technique"]})')
print(f'MTTD min : {df_kpi["mttd"].min()}s ({df_kpi.loc[df_kpi["mttd"].idxmin(), "technique"]})')
print(f'\nMTTD moyen Session 1 : {df_kpi[df_kpi.session==1]["mttd"].mean():.1f}s')
print(f'MTTD moyen Session 2 : {df_kpi[df_kpi.session==2]["mttd"].mean():.1f}s')
print(f'MTTD moyen Session 3 : {df_kpi[df_kpi.session==3]["mttd"].mean():.1f}s')

In [ ]:
# Visualisation MTTD par scénario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3' if s==1 else '#4CAF50' if s==2 else '#FF5722' 
          for s in df_kpi['session']]

# Bar chart MTTD
axes[0].bar(df_kpi['sc'], df_kpi['mttd'], color=colors, edgecolor='white')
axes[0].axhline(y=300, color='red', linestyle='--', label='Objectif max (300s)')
axes[0].axhline(y=df_kpi['mttd'].mean(), color='orange', linestyle='--', 
                label=f'Moyenne ({df_kpi["mttd"].mean():.1f}s)')
axes[0].set_title('MTTD par Scénario Purple Team', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Scénario')
axes[0].set_ylabel('MTTD (secondes)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# MTTD par session
session_mttd = df_kpi.groupby('session')['mttd'].mean()
axes[1].bar(['Session 1\n(7 SC)', 'Session 2\n(2 SC)', 'Session 3\n(3 SC)'],
            session_mttd.values,
            color=['#2196F3', '#4CAF50', '#FF5722'],
            edgecolor='white')
axes[1].axhline(y=300, color='red', linestyle='--', label='Objectif max (300s)')
axes[1].set_title('MTTD Moyen par Session', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MTTD moyen (secondes)')
axes[1].legend()
for i, v in enumerate(session_mttd.values):
    axes[1].text(i, v + 1, f'{v:.1f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../../docs/screenshots/kpi-mttd-chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé.')

## Conclusion du Hunting

| Hunt | Hypothèse | Résultat | Action |
|------|-----------|----------|--------|
| H1 | PowerShell suspect | Confirmé (SC-01, SC-07) | Règles 100131/100121 opérationnelles |
| H2 | Brute Force AD | Confirmé (SC-02) | Règle 100111 opérationnelle |
| H3 | Admin Shares | Confirmé (SC-04) | Volume élevé → agrégation ajoutée |
| H4 | Persistence | Confirmé (SC-07, SC-10, SC-12) | Règles 92302/60642/92004 opérationnelles |
| H5 | LSASS Dump | Confirmé (SC-05) | Règle 100121 niveau 14 opérationnelle |

**Toutes les hypothèses confirmées. Aucune technique non détectée identifiée.**